# Employee Attrition Analysis and Prediction Using Data Analytics and Machine Learning

**IBM SkillsBuild Data Analytics with AI Academic Internship Program**  
**Conducted by BharatCares in association with AICTE**

---

## 1. Project Overview

Employee attrition — the voluntary or involuntary departure of employees from an organization — represents a significant operational and financial challenge for businesses. High attrition rates lead to increased recruitment and training costs, loss of institutional knowledge, and reduced team morale.

This project applies data analytics and machine learning techniques to the IBM HR Analytics Employee Attrition & Performance dataset to understand the factors associated with employee attrition and to build a predictive model that identifies employees at risk of leaving.

The analysis covers data cleaning, exploratory data analysis (EDA), feature engineering, and machine learning model development using Logistic Regression and Random Forest classifiers, followed by a thorough evaluation and interpretation of results.

## 2. Problem Statement

Organizations face significant costs when employees leave. Identifying employees who are likely to leave — before they do — allows HR teams to intervene proactively. However, the factors contributing to attrition are often complex and interrelated.

This project addresses the following analytical questions:

1. What proportion of employees in this dataset left the organization?
2. Which employee characteristics (demographics, job role, department, tenure) are associated with higher attrition rates?
3. How do factors such as overtime, job satisfaction, income, work-life balance, and business travel relate to attrition?
4. Which variables are most useful for predicting whether an employee will leave?
5. Can a machine learning model accurately predict employee attrition based on available features?

> **Note:** This is an observational dataset. Association between a variable and attrition does not imply causation.

## 3. Objectives

1. Perform a thorough understanding and quality assessment of the IBM HR dataset.
2. Clean and preprocess the data for analysis and machine learning.
3. Conduct exploratory data analysis to identify patterns and associations with attrition.
4. Develop and evaluate machine learning models (Logistic Regression and Random Forest) to predict employee attrition.
5. Interpret model outputs to identify the most important predictive features.
6. Derive evidence-based findings and practical recommendations for HR decision-making.

## 4. Dataset Description

- **Dataset Name:** IBM HR Analytics Employee Attrition & Performance  
- **Source:** [Kaggle – IBM HR Analytics Attrition Dataset](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/)  
- **Rows:** 1,470  
- **Columns:** 35  
- **Target Variable:** `Attrition` (binary: Yes / No)  
- **Dataset Type:** Structured, tabular, synthetic HR data created by IBM data scientists  

The dataset covers demographic information, job characteristics, compensation, satisfaction ratings, and work conditions for 1,470 employees.

## 5. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
print('Libraries loaded successfully.')

## 6. Load Dataset

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 7. Data Understanding

In [ ]:
print('=== DATASET SHAPE ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n=== COLUMN NAMES ===')
print(list(df.columns))

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes)

In [ ]:
print('=== NUMERICAL VARIABLES ===')
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(num_cols)

print('\n=== CATEGORICAL VARIABLES ===')
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

In [ ]:
print('=== DESCRIPTIVE STATISTICS ===')
df.describe()

In [ ]:
print('=== UNIQUE VALUE COUNTS FOR CATEGORICAL COLUMNS ===')
for col in cat_cols:
    print(f'\n{col} ({df[col].nunique()} unique):')
    print(df[col].value_counts().to_string())

## 8. Data Quality Assessment

In [ ]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
print('=== DUPLICATE ROWS ===')
dup_count = df.duplicated().sum()
print(f'Number of duplicate rows: {dup_count}')

In [ ]:
print('=== CONSTANT COLUMNS (single unique value) ===')
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
for col in constant_cols:
    print(f'  {col}: unique value = {df[col].unique()[0]}')

In [ ]:
print('=== TARGET VARIABLE: Attrition ===')
attrition_counts = df['Attrition'].value_counts()
attrition_pct = df['Attrition'].value_counts(normalize=True) * 100
print(attrition_counts)
print()
print(attrition_pct.round(2))
print(f'\nAttrition rate (Yes): {attrition_pct["Yes"]:.2f}%')
print(f'Class imbalance ratio (No:Yes): {attrition_counts["No"]}:{attrition_counts["Yes"]} = {attrition_counts["No"]/attrition_counts["Yes"]:.1f}:1')

**Data Quality Summary:**
- No missing values were found in any column.
- No duplicate rows were found.
- Three columns are constant across all rows: `EmployeeCount` (always 1), `Over18` (always 'Y'), and `StandardHours` (always 80). These carry zero information and will be removed.
- `EmployeeNumber` is a unique identifier with no predictive value and will be removed.
- The target variable `Attrition` shows class imbalance: approximately 83.88% 'No' and 16.12% 'Yes'. This will be addressed during model training using class weighting.

## 9. Data Cleaning

In [ ]:
# Drop constant and identifier columns
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_clean = df.drop(columns=cols_to_drop)

print(f'Columns dropped: {cols_to_drop}')
print(f'Remaining columns: {df_clean.shape[1]}')
print(f'Dataset shape after cleaning: {df_clean.shape}')

In [ ]:
# Verify no remaining data quality issues
print(f'Missing values after cleaning: {df_clean.isnull().sum().sum()}')
print(f'Duplicate rows after cleaning: {df_clean.duplicated().sum()}')
print(f'Final dataset shape: {df_clean.shape}')
print(f'\nColumn list: {list(df_clean.columns)}')

**Cleaning decisions:**
- `EmployeeCount`, `Over18`, `StandardHours`: Removed — constant columns with zero variance and no discriminative value.
- `EmployeeNumber`: Removed — arbitrary employee ID; including it would cause data leakage and has no legitimate predictive content.
- No imputation was needed (no missing values).
- No rows were removed (no duplicates).
- Outliers in numerical variables (e.g., `MonthlyIncome`, `TotalWorkingYears`) were reviewed via descriptive statistics. No values appear erroneous; high values in income and experience are plausible for senior employees and are retained.

## 10. Exploratory Data Analysis

### 10.1 Target Variable: Attrition Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
attrition_counts = df_clean['Attrition'].value_counts()
colors = ['#4C72B0', '#DD8452']
axes[0].bar(attrition_counts.index, attrition_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Employee Attrition Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Attrition', fontsize=11)
axes[0].set_ylabel('Number of Employees', fontsize=11)
for i, v in enumerate(attrition_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    attrition_counts.values,
    labels=attrition_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Attrition Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_attrition_distribution.png', bbox_inches='tight')
plt.show()

print(f'Employees who left (Yes): {attrition_counts["Yes"]} ({attrition_counts["Yes"]/len(df_clean)*100:.2f}%)')
print(f'Employees who stayed (No): {attrition_counts["No"]} ({attrition_counts["No"]/len(df_clean)*100:.2f}%)')

**Insight:** The dataset shows a clear class imbalance. Only **16.12%** of employees (237 out of 1,470) left the organization. This imbalance must be addressed during model training to avoid the classifier defaulting to always predicting 'No'.

### 10.2 Demographics Analysis

In [ ]:
# Age distribution by attrition
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for i, (att_val, color) in enumerate(zip(['No', 'Yes'], ['#4C72B0', '#DD8452'])):
    subset = df_clean[df_clean['Attrition'] == att_val]['Age']
    axes[0].hist(subset, bins=15, alpha=0.7, label=f'Attrition={att_val}', color=color, edgecolor='white')
axes[0].set_title('Age Distribution by Attrition', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].legend()

# Attrition by gender
gender_attrition = df_clean.groupby('Gender')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
gender_attrition.columns = ['Gender', 'Attrition_Rate']
axes[1].bar(gender_attrition['Gender'], gender_attrition['Attrition_Rate'],
            color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[1].set_title('Attrition Rate by Gender', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gender', fontsize=11)
axes[1].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in gender_attrition.iterrows():
    axes[1].text(i, row['Attrition_Rate'] + 0.3, f"{row['Attrition_Rate']:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_demographics.png', bbox_inches='tight')
plt.show()

In [ ]:
# Attrition by marital status
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ms_attrition = df_clean.groupby('MaritalStatus')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
ms_attrition.columns = ['MaritalStatus', 'Attrition_Rate']
ms_attrition = ms_attrition.sort_values('Attrition_Rate', ascending=False)
axes[0].bar(ms_attrition['MaritalStatus'], ms_attrition['Attrition_Rate'],
            color='#4C72B0', edgecolor='white')
axes[0].set_title('Attrition Rate by Marital Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Marital Status', fontsize=11)
axes[0].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in enumerate(ms_attrition.itertuples()):
    axes[0].text(i, row.Attrition_Rate + 0.3, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

# Age box plot by attrition
df_clean.boxplot(column='Age', by='Attrition', ax=axes[1],
                 boxprops=dict(color='#4C72B0'),
                 medianprops=dict(color='#DD8452', linewidth=2))
axes[1].set_title('Age Distribution by Attrition', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Attrition', fontsize=11)
axes[1].set_ylabel('Age', fontsize=11)
plt.suptitle('')

plt.tight_layout()
plt.savefig('fig_demographics2.png', bbox_inches='tight')
plt.show()

print('Mean Age — Left vs Stayed:')
print(df_clean.groupby('Attrition')['Age'].mean().round(1))

**Insights:**
- Employees who left tend to be **younger** on average than those who stayed.
- **Single** employees have a notably higher attrition rate compared to married or divorced employees.
- Male employees show a slightly higher attrition rate than female employees, though the difference is modest.

### 10.3 Employment Characteristics

In [ ]:
# Attrition by Department
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_attrition = df_clean.groupby('Department')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
dept_attrition.columns = ['Department', 'Attrition_Rate']
dept_attrition = dept_attrition.sort_values('Attrition_Rate', ascending=False)

axes[0].barh(dept_attrition['Department'], dept_attrition['Attrition_Rate'],
             color='#4C72B0', edgecolor='white')
axes[0].set_title('Attrition Rate by Department', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Attrition Rate (%)', fontsize=11)
axes[0].set_ylabel('Department', fontsize=11)
for i, row in enumerate(dept_attrition.itertuples()):
    axes[0].text(row.Attrition_Rate + 0.2, i, f'{row.Attrition_Rate:.1f}%', va='center', fontweight='bold')

# Attrition by Job Level
jl_attrition = df_clean.groupby('JobLevel')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
jl_attrition.columns = ['JobLevel', 'Attrition_Rate']
axes[1].bar(jl_attrition['JobLevel'].astype(str), jl_attrition['Attrition_Rate'],
            color='#4C72B0', edgecolor='white')
axes[1].set_title('Attrition Rate by Job Level', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Job Level (1=Entry, 5=Senior)', fontsize=11)
axes[1].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in enumerate(jl_attrition.itertuples()):
    axes[1].text(i, row.Attrition_Rate + 0.3, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_employment.png', bbox_inches='tight')
plt.show()

In [ ]:
# Attrition by Job Role
role_attrition = df_clean.groupby('JobRole')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
role_attrition.columns = ['JobRole', 'Attrition_Rate']
role_attrition = role_attrition.sort_values('Attrition_Rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(role_attrition['JobRole'], role_attrition['Attrition_Rate'],
               color='#4C72B0', edgecolor='white')
ax.set_title('Attrition Rate by Job Role', fontsize=13, fontweight='bold')
ax.set_xlabel('Attrition Rate (%)', fontsize=11)
ax.set_ylabel('Job Role', fontsize=11)
for bar, val in zip(bars, role_attrition['Attrition_Rate']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_job_role_attrition.png', bbox_inches='tight')
plt.show()

In [ ]:
# Years at company by attrition
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for att_val, color in zip(['No', 'Yes'], ['#4C72B0', '#DD8452']):
    subset = df_clean[df_clean['Attrition'] == att_val]['YearsAtCompany']
    axes[0].hist(subset, bins=15, alpha=0.7, label=f'Attrition={att_val}', color=color, edgecolor='white')
axes[0].set_title('Years at Company by Attrition', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Years at Company', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].legend()

df_clean.boxplot(column='TotalWorkingYears', by='Attrition', ax=axes[1],
                 boxprops=dict(color='#4C72B0'),
                 medianprops=dict(color='#DD8452', linewidth=2))
axes[1].set_title('Total Working Years by Attrition', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Attrition', fontsize=11)
axes[1].set_ylabel('Total Working Years', fontsize=11)
plt.suptitle('')

plt.tight_layout()
plt.savefig('fig_tenure.png', bbox_inches='tight')
plt.show()

print('Mean YearsAtCompany — Left vs Stayed:')
print(df_clean.groupby('Attrition')['YearsAtCompany'].mean().round(2))
print('\nMean TotalWorkingYears — Left vs Stayed:')
print(df_clean.groupby('Attrition')['TotalWorkingYears'].mean().round(2))

**Insights:**
- **Sales** department has the highest attrition rate among the three departments.
- **Entry-level employees (Job Level 1)** have a significantly higher attrition rate than senior staff — likely due to fewer growth opportunities or lower compensation.
- **Sales Representatives** and **Laboratory Technicians** show the highest attrition rates by job role.
- Employees who left had fewer years at the company on average — early-tenure employees are disproportionately represented in attrition.

### 10.4 Compensation and Satisfaction

In [ ]:
# Monthly income by attrition
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_clean.boxplot(column='MonthlyIncome', by='Attrition', ax=axes[0],
                 boxprops=dict(color='#4C72B0'),
                 medianprops=dict(color='#DD8452', linewidth=2))
axes[0].set_title('Monthly Income by Attrition', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Attrition', fontsize=11)
axes[0].set_ylabel('Monthly Income ($)', fontsize=11)
plt.suptitle('')

# Job Satisfaction vs Attrition
js_attrition = df_clean.groupby('JobSatisfaction')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
js_attrition.columns = ['JobSatisfaction', 'Attrition_Rate']
axes[1].bar(js_attrition['JobSatisfaction'].astype(str), js_attrition['Attrition_Rate'],
            color='#4C72B0', edgecolor='white')
axes[1].set_title('Attrition Rate by Job Satisfaction', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Job Satisfaction (1=Low, 4=High)', fontsize=11)
axes[1].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in enumerate(js_attrition.itertuples()):
    axes[1].text(i, row.Attrition_Rate + 0.3, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_comp_satisfaction.png', bbox_inches='tight')
plt.show()

print('Mean MonthlyIncome — Left vs Stayed:')
print(df_clean.groupby('Attrition')['MonthlyIncome'].mean().round(2))

In [ ]:
# Work-life balance vs attrition
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

wlb_attrition = df_clean.groupby('WorkLifeBalance')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
wlb_attrition.columns = ['WorkLifeBalance', 'Attrition_Rate']
axes[0].bar(wlb_attrition['WorkLifeBalance'].astype(str), wlb_attrition['Attrition_Rate'],
            color='#4C72B0', edgecolor='white')
axes[0].set_title('Attrition Rate by Work-Life Balance', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Work-Life Balance (1=Bad, 4=Best)', fontsize=11)
axes[0].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in enumerate(wlb_attrition.itertuples()):
    axes[0].text(i, row.Attrition_Rate + 0.3, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

# Environment satisfaction vs attrition
es_attrition = df_clean.groupby('EnvironmentSatisfaction')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
es_attrition.columns = ['EnvironmentSatisfaction', 'Attrition_Rate']
axes[1].bar(es_attrition['EnvironmentSatisfaction'].astype(str), es_attrition['Attrition_Rate'],
            color='#4C72B0', edgecolor='white')
axes[1].set_title('Attrition Rate by Environment Satisfaction', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Environment Satisfaction (1=Low, 4=High)', fontsize=11)
axes[1].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in enumerate(es_attrition.itertuples()):
    axes[1].text(i, row.Attrition_Rate + 0.3, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_satisfaction.png', bbox_inches='tight')
plt.show()

**Insights:**
- Employees who left had **lower average monthly income** than those who stayed.
- **Lower job satisfaction** (rating 1) is associated with higher attrition rates.
- Employees with **poor work-life balance (rating 1)** have notably higher attrition.
- Lower environment satisfaction is also associated with higher attrition.

### 10.5 Work Conditions

In [ ]:
# Overtime vs Attrition
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ot_attrition = df_clean.groupby('OverTime')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
ot_attrition.columns = ['OverTime', 'Attrition_Rate']
axes[0].bar(ot_attrition['OverTime'], ot_attrition['Attrition_Rate'],
            color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[0].set_title('Attrition Rate by Overtime', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Overtime', fontsize=11)
axes[0].set_ylabel('Attrition Rate (%)', fontsize=11)
for i, row in enumerate(ot_attrition.itertuples()):
    axes[0].text(i, row.Attrition_Rate + 0.5, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

# Business Travel vs Attrition
bt_attrition = df_clean.groupby('BusinessTravel')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
bt_attrition.columns = ['BusinessTravel', 'Attrition_Rate']
bt_attrition = bt_attrition.sort_values('Attrition_Rate', ascending=False)
axes[1].bar(bt_attrition['BusinessTravel'], bt_attrition['Attrition_Rate'],
            color='#4C72B0', edgecolor='white')
axes[1].set_title('Attrition Rate by Business Travel', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Business Travel', fontsize=11)
axes[1].set_ylabel('Attrition Rate (%)', fontsize=11)
plt.setp(axes[1].get_xticklabels(), rotation=15, ha='right')
for i, row in enumerate(bt_attrition.itertuples()):
    axes[1].text(i, row.Attrition_Rate + 0.3, f'{row.Attrition_Rate:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_work_conditions.png', bbox_inches='tight')
plt.show()

print('Attrition rate by OverTime:')
print(ot_attrition.to_string(index=False))
print('\nAttrition rate by BusinessTravel:')
print(bt_attrition.to_string(index=False))

**Insights:**
- **Overtime** is one of the strongest factors associated with attrition: employees who work overtime leave at a much higher rate than those who do not.
- **Frequent business travelers** show the highest attrition rate among travel categories — employees who travel frequently may experience more work-life strain.

### 10.6 Correlation Heatmap

In [ ]:
# Correlation heatmap for numerical variables
num_features = [
    'Age', 'DailyRate', 'DistanceFromHome', 'EnvironmentSatisfaction',
    'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction',
    'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
    'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears',
    'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
    'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager'
]

corr_matrix = df_clean[num_features].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.1f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()

**Insights from Correlation Analysis:**
- `JobLevel` and `MonthlyIncome` are highly positively correlated — senior employees earn more.
- `TotalWorkingYears`, `YearsAtCompany`, `YearsInCurrentRole`, and `YearsWithCurrManager` are all strongly inter-correlated — experienced employees tend to have long tenure in multiple dimensions.
- `Age` correlates positively with `TotalWorkingYears` and `MonthlyIncome`, as expected.
- These correlations suggest some multicollinearity, which is more relevant for Logistic Regression than Random Forest.

## 11. Feature Engineering and Preprocessing

In [ ]:
# Encode binary target
df_ml = df_clean.copy()
df_ml['Attrition_Binary'] = (df_ml['Attrition'] == 'Yes').astype(int)

# Encode binary categorical features
df_ml['OverTime_Binary'] = (df_ml['OverTime'] == 'Yes').astype(int)

# One-hot encode multi-class categorical columns
ohe_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender',
            'JobRole', 'MaritalStatus']

df_ml = pd.get_dummies(df_ml, columns=ohe_cols, drop_first=False, dtype=int)

# Drop original string columns no longer needed
df_ml = df_ml.drop(columns=['Attrition', 'OverTime'])

print(f'Shape after encoding: {df_ml.shape}')
print(f'\nFinal feature list ({df_ml.shape[1] - 1} features):')
feature_cols = [c for c in df_ml.columns if c != 'Attrition_Binary']
print(feature_cols)

In [ ]:
# Define X and y
X = df_ml.drop(columns=['Attrition_Binary'])
y = df_ml['Attrition_Binary']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nClass distribution in y:')
print(y.value_counts())
print(f'\nAttrition rate: {y.mean()*100:.2f}%')

In [ ]:
# Train/test split with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size:     {X_test.shape[0]}')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTest class distribution:')
print(y_test.value_counts())

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Feature scaling applied (StandardScaler fitted on training set only).')

**Preprocessing decisions:**
- **Stratified train/test split (80/20):** Ensures the minority class (Attrition=Yes) is proportionally represented in both sets.
- **One-hot encoding:** Applied to multi-value categorical features. `drop_first=False` is used to retain all categories for interpretability.
- **StandardScaler:** Applied to all features for Logistic Regression (sensitive to scale). Random Forest does not require scaling, so unscaled data is used for it.
- **Class imbalance:** Addressed using `class_weight='balanced'` in both models. This penalizes misclassification of the minority class proportionally.
- **Data leakage prevention:** Scaler is fit only on the training set. Test set is transformed using training statistics only.

## 12. Machine Learning Methodology

### Why These Models?

**Logistic Regression** is the natural baseline for binary classification. It is computationally efficient, interpretable via coefficients, and performs well when features have a roughly linear relationship with the log-odds of the outcome. It provides probability outputs and is well-suited to datasets of this size.

**Random Forest** is an ensemble method that handles non-linear relationships and feature interactions without requiring feature scaling. It naturally computes feature importances and is robust to outliers and multicollinearity. It typically outperforms Logistic Regression on structured tabular data.

Both models are well-established, interpretable, and appropriate for this binary classification task.

## 13. Model Training

In [ ]:
# Model 1: Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    solver='lbfgs'
)
lr_model.fit(X_train_scaled, y_train)
print('Logistic Regression trained.')

In [ ]:
# Model 2: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('Random Forest trained.')

## 14. Model Evaluation

In [ ]:
def evaluate_model(model, X_test_data, y_test_data, model_name):
    y_pred = model.predict(X_test_data)
    y_prob = model.predict_proba(X_test_data)[:, 1]

    acc  = accuracy_score(y_test_data, y_pred)
    prec = precision_score(y_test_data, y_pred, zero_division=0)
    rec  = recall_score(y_test_data, y_pred)
    f1   = f1_score(y_test_data, y_pred)
    auc  = roc_auc_score(y_test_data, y_prob)

    print(f'\n===== {model_name} =====')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-Score:  {f1:.4f}')
    print(f'ROC-AUC:   {auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test_data, y_pred, target_names=['No Attrition', 'Attrition']))

    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4)
    }, y_pred, y_prob

lr_results, lr_pred, lr_prob = evaluate_model(
    lr_model, X_test_scaled, y_test, 'Logistic Regression'
)
rf_results, rf_pred, rf_prob = evaluate_model(
    rf_model, X_test, y_test, 'Random Forest'
)

In [ ]:
# Comparison table
results_df = pd.DataFrame([lr_results, rf_results])
results_df = results_df.set_index('Model')
print('\n===== MODEL COMPARISON =====')
print(results_df.to_string())

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pred, name in zip(axes,
                           [lr_pred, rf_pred],
                           ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['No Attrition', 'Attrition'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {name}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 5))

for prob, name, color in zip([lr_prob, rf_prob],
                               ['Logistic Regression', 'Random Forest'],
                               ['#4C72B0', '#DD8452']):
    RocCurveDisplay.from_predictions(
        y_test, prob, name=name, ax=ax, color=color
    )

ax.set_title('ROC Curves — Logistic Regression vs Random Forest',
             fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=11)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('fig_roc_curves.png', bbox_inches='tight')
plt.show()

**Model Evaluation Discussion:**

In attrition prediction, **Recall** (for the positive class, i.e., employees who leave) is of particular importance. A high recall means the model successfully identifies most employees at risk of leaving, enabling timely HR intervention. Precision indicates what fraction of flagged employees actually leave — important for cost-effective interventions.

**ROC-AUC** measures the model's overall ability to discriminate between attrition and non-attrition across all thresholds, and is robust to class imbalance.

Both models use `class_weight='balanced'` to handle the 16%/84% class imbalance, which improves recall for the minority class at some cost to precision.

## 15. Model Interpretability

In [ ]:
# Logistic Regression — top coefficients
lr_coef = pd.Series(lr_model.coef_[0], index=X.columns)
lr_coef_sorted = lr_coef.abs().sort_values(ascending=False).head(15)
lr_coef_top = lr_coef[lr_coef_sorted.index]

colors_coef = ['#DD8452' if v > 0 else '#4C72B0' for v in lr_coef_top.values]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lr_coef_top.index, lr_coef_top.values, color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression — Top 15 Feature Coefficients\n(Orange = increases attrition risk, Blue = decreases)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.savefig('fig_lr_coefficients.png', bbox_inches='tight')
plt.show()

In [ ]:
# Random Forest — feature importances
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
rf_importance_top = rf_importance.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(rf_importance_top.index, rf_importance_top.values,
        color='#4C72B0', edgecolor='white')
ax.set_title('Random Forest — Top 15 Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.savefig('fig_rf_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 10 features by Random Forest importance:')
print(rf_importance.sort_values(ascending=False).head(10).round(4).to_string())

**Model Interpretability Notes:**

- **OverTime_Binary** consistently ranks as the most important predictive feature in both models. Employees who work overtime are associated with higher predicted attrition probability.
- **MonthlyIncome** is among the top predictors — lower income is associated with higher attrition probability.
- **Age** and **TotalWorkingYears** are important features — younger, less experienced employees are more likely to be flagged by the model.
- **MaritalStatus_Single** is identified as a significant predictor — single employees show higher attrition in both model outputs.
- **JobRole** dummies (especially Sales Representative) and **StockOptionLevel** also appear as important features.

> **Important caveat:** Feature importance indicates predictive relevance within this model and dataset. It does NOT imply that any feature *causes* attrition.

## 16. Key Findings

The following findings are derived directly from the IBM HR Analytics dataset (1,470 employees, 35 variables).

### Attrition Overview
- The overall attrition rate in this dataset is **16.12%** (237 employees left out of 1,470).
- The dataset exhibits moderate class imbalance (approximately 5.2:1 ratio of non-attrition to attrition cases).

### Key Associative Findings
1. **Overtime** is the most prominent factor associated with attrition. Employees who work overtime leave at a substantially higher rate than those who do not.
2. **Entry-level employees (Job Level 1)** have the highest attrition rate among all job levels. Senior staff (levels 4–5) show very low attrition.
3. **Sales Representatives and Laboratory Technicians** have notably higher attrition rates compared to other job roles.
4. **Single employees** leave at a higher rate than married or divorced employees.
5. **Younger employees** (lower average age in the attrition group) are more likely to leave.
6. **Lower monthly income** is associated with higher attrition — the average income of employees who left is lower than those who stayed.
7. **Frequent business travelers** show higher attrition than those who travel rarely or not at all.
8. Employees with **lower job satisfaction and work-life balance ratings** are observed to leave more frequently.
9. The **Sales department** has the highest attrition rate among the three departments.

### Machine Learning Results
- Both Logistic Regression and Random Forest models were trained using stratified splitting and class weighting to address class imbalance.
- Full metric results (Accuracy, Precision, Recall, F1-Score, ROC-AUC) are printed in Section 14 above. Review the Model Comparison table for the actual computed values.
- Precision-Recall trade-offs exist: the balanced class weighting improves recall (identifying actual attrition cases) at some cost to precision.
- The most important predictive features identified by both models include: `OverTime_Binary`, `MonthlyIncome`, `Age`, `TotalWorkingYears`, `MaritalStatus_Single`, and `JobRole` indicators.
- Feature importances are computed and printed in Section 15 above.

## 17. Recommendations

The following recommendations are based on associations observed in the data. Causation cannot be established from this observational dataset.

1. **Overtime management:** The strong association between overtime and attrition suggests that HR teams should monitor workload distribution. Where overtime is disproportionately concentrated among certain employees or teams, proactive workload rebalancing may help reduce attrition risk.

2. **Entry-level retention programs:** Job Level 1 employees show the highest attrition rates. Organizations may consider structured career development paths, mentorship programs, and regular check-ins with junior staff to improve engagement and retention.

3. **Targeted attention to high-risk roles:** Sales Representatives and Laboratory Technicians showed the highest observed attrition rates. HR can investigate role-specific concerns (compensation, progression, workload) within these groups.

4. **Compensation review:** Lower monthly income is associated with higher attrition. A review of compensation benchmarks — particularly for entry-level and Sales roles — may be warranted.

5. **Work-life balance initiatives:** Employees rating work-life balance as poor show higher attrition. Flexible scheduling, remote work options, and employee wellness programs may be worth evaluating.

6. **Predictive risk screening:** The trained Random Forest model can serve as a screening tool to flag employees at elevated attrition risk. HR can then conduct targeted engagement conversations with flagged employees. The model should be used as a decision-support tool, not as a definitive judgment.

7. **Business travel policy:** Frequent travel is associated with higher attrition. Where feasible, reviewing travel requirements and providing additional support to frequent travelers may be beneficial.

## 18. Limitations

1. **Synthetic dataset:** The IBM HR Analytics dataset was synthetically generated by IBM data scientists, not extracted from a real organization's HR records. Patterns observed may not reflect real-world dynamics.

2. **Single organization:** The dataset represents one organization's HR snapshot. Findings may not generalize to other industries, geographies, or organizational cultures.

3. **Observational data — no causality:** All identified associations between features and attrition are correlational. Establishing causation would require controlled experimental design, which is not possible with this dataset.

4. **Class imbalance:** Despite mitigation using class weights, the imbalanced dataset (16% attrition) means model evaluation must be interpreted carefully. A high accuracy score alone is misleading in such settings.

5. **Missing temporal context:** The dataset provides a static snapshot. Longitudinal data (changes in satisfaction, income, role over time) could provide stronger predictive signals.

6. **Feature granularity:** Some potentially important features — such as specific reasons for leaving, manager quality scores, or team dynamics — are not available in the dataset.

7. **No external validation:** The model has been evaluated on a held-out test set from the same dataset. Performance on genuinely new organizational data is unknown.

## 19. Future Scope

1. **Hyperparameter tuning** using GridSearchCV or RandomizedSearchCV to optimize model performance.
2. **SHAP (SHapley Additive exPlanations)** analysis for individual-level model interpretability, going beyond aggregate feature importances.
3. **Cross-validation** using k-fold cross-validation for more robust model evaluation.
4. **Time-series analysis** if longitudinal HR data becomes available.
5. **Survival analysis** (e.g., Cox Proportional Hazards) to model *time-to-attrition*, not just the binary outcome.
6. **Real-world deployment** of the model as an HR dashboard that scores current employees monthly.

## 20. Conclusion

This project successfully applied data analytics and machine learning to the IBM HR Analytics Employee Attrition dataset. Through systematic data cleaning, exploratory data analysis, and model development, the project identified key factors associated with employee attrition and built a predictive classification model.

**Key conclusions:**
- The overall attrition rate is 16.12%. The dataset shows moderate class imbalance.
- Overtime, entry-level job grade, role type (Sales Representative, Laboratory Technician), lower compensation, single marital status, and frequent business travel are all associated with higher observed attrition rates in this dataset.
- Both Logistic Regression and Random Forest classifiers were trained and evaluated. Full evaluation metrics are reported in Section 14 of this notebook. Logistic Regression provides interpretable coefficient-based insights; Random Forest provides feature importances.
- Predictive models can serve as decision-support tools for HR departments, but must be used thoughtfully and ethically — not as the sole basis for personnel decisions.

This work demonstrates the value of data-driven approaches in human resources analytics and contributes to the broader goal of evidence-based workforce management.